# ⚗️ BioPath Reaction Transformer Demo
### *Shortest-Path Distance (SPD) Topological Attention & Oncometabolite Flux Perturbation*

Inspired by **Ying et al. (NeurIPS 2021, Graphormer)** and **Schwaller et al. (Nature Machine Intelligence 2019)**, this notebook demonstrates:
1. Graphormer Multi-Head Attention biased by pairwise shortest-path distances across human metabolic networks.
2. Predicting metabolic reaction flux and oncometabolite accumulation.
3. Simulating the oncogenic **IDH1 R132H** neomorphic mutation driving **2-hydroxyglutarate (2-HG)** surge in glioblastoma and AML.

In [ ]:
import os
import sys
sys.path.append('..')

import torch
import numpy as np
import matplotlib.pyplot as plt

from src.config import METABOLITE_NAMES, ENZYME_NAMES, MODEL_CONFIG
from src.metabolic_graph import MetabolicGraph
from src.metabolic_tokenizer import MetabolicTokenizer
from src.biopath_transformer import BioPathTransformerModel
from src.flux_perturbation import MetabolicFluxPerturbation
from src.visualizer import BioPathVisualizer

print("PyTorch Version:", torch.__version__)
print(f"Registered Metabolites: {len(METABOLITE_NAMES)}")
print(f"Registered Enzymes:     {len(ENZYME_NAMES)}")

## 1. Inspect Topological Shortest-Path Distances in TCA Cycle

In [ ]:
graph = MetabolicGraph(max_spd=8)
spd = graph.get_spd_matrix()

sample_nodes = ["Glucose", "Pyruvate", "Citrate", "Alpha_Ketoglutarate", "Succinate", "2_Hydroxyglutarate"]
for u in sample_nodes:
    u_id = graph.metabolite_to_id[u]
    dists = [f"{v}: {spd[u_id, graph.metabolite_to_id[v]]}" for v in sample_nodes if v != u]
    print(f"{u:<20} -> " + ", ".join(dists[:3]))

## 2. Load the Pre-Trained BioPath Model

In [ ]:
tokenizer = MetabolicTokenizer(max_nodes=MODEL_CONFIG['max_graph_nodes'], max_spd=MODEL_CONFIG['max_spd_distance'])
model = BioPathTransformerModel(MODEL_CONFIG)

ckpt_path = '../checkpoints/best_biopath_model.pt'
if os.path.exists(ckpt_path):
    model.load_state_dict(torch.load(ckpt_path, map_location='cpu'))
    print("Loaded trained weights from:", ckpt_path)
else:
    print("Using randomly initialized model (run train_and_evaluate.py to train!)")
model.eval()

## 3. Simulate IDH1 R132H Oncometabolite Surge

In [ ]:
pert = MetabolicFluxPerturbation(model, tokenizer)
res = pert.simulate_idh1_r132h()

print("Oncogenic Alteration:      ", res["alteration"])
print("Wild-Type TCA Flux:        ", f"{res['wt_flux']:.3f} umol/min")
print("Mutant Shunted Flux:       ", f"{res['mut_flux']:.3f} umol/min")
print("Forward Flux Inhibition:   ", f"{res['flux_inhibition_pct']:.1f}%")
print("2-HG Oncometabolite Surge: ", f"{res['oncometabolite_fold_surge']:.2f}x fold increase")
print("Clinical Mechanism:        ", res["clinical_annotation"]["mechanism"])

## 4. Visualize Topological SPD-Biased Attention Heatmap

In [ ]:
viz = BioPathVisualizer(output_dir='../figures')
tca_sample = ["Citrate", "Isocitrate", "Alpha_Ketoglutarate", "Succinate", "Fumarate", "Malate", "Oxaloacetate"]
ids, mask, spd_sub = tokenizer.encode_pathway(tca_sample)

attn_res = model.extract_attention(ids.unsqueeze(0), spd_sub.unsqueeze(0), attention_mask=mask.unsqueeze(0))
last_layer_attn = attn_res['layer_attention_weights'][-1, 0].mean(dim=0).cpu().numpy()

fig_path = viz.plot_spd_attention_heatmap(
    attn_matrix=last_layer_attn,
    node_labels=tca_sample,
    filename='demo_spd_attention_heatmap.png'
)

from IPython.display import Image
Image(fig_path)